# Converting Migration Torques to SpaceHub Acceleration Vectors

This notebook derives the acceleration vectors needed for SpaceHub's `add_acc_to` interface, given scalar migration torque prescriptions. Three cases are treated:

1. **Generic torque** $\Gamma(\Sigma, R, \Omega, \ldots)$ in the circular orbit limit
2. **Type I migration torques** (Jiménez & Masset / Paardekooper prescriptions)
3. **Eccentric orbit evolution** from inverse timescales $\tau_a^{-1}$, $\tau_e^{-1}$ (Fairbairn & Rafikov 2025)

---
## Meeting Notes Transcription (2/5/2026)

### Above the blue lines — Disk model power-law parameterization

$$\Sigma(r) = \Sigma_0 \left(\frac{r}{r_0}\right)^{-p}, \qquad c_s(r) = c_{s,0}\left(\frac{r}{r_0}\right)^{-q/2}$$

$$\frac{d\ln\Sigma}{d\ln R} = -p, \qquad \frac{d\ln c_s}{d\ln R} = -\frac{q}{2}$$

$$\rho(R,z) = \rho(R,0)\,e^{-z^2/(2H^2(R))} \quad \longleftarrow \rho_g(R) \text{ (midplane density)}$$

These define the radial profiles and vertical density structure of the disk. The log-gradients $-p$ and $-q/2$ appear in the torque coefficients $C_I$ (see the Jiménez & Masset / Paardekooper prescriptions).

---

### Between the blue lines — Converting torque $\Gamma$ to $\dot{a}$

Starting point: the torque is the time derivative of angular momentum.

$$\Gamma = \frac{dJ}{dt} \longrightarrow \dot{a}$$

For a circular Keplerian orbit the angular momentum is $J = m\sqrt{GMa}$, so

$$\Gamma = \frac{dJ}{dt} = m\sqrt{GM}\;\frac{\dot{a}}{2\sqrt{a}} = \frac{m}{2}\sqrt{\frac{GM}{a}}\;\dot{a}$$

Solving for the semi-major axis evolution:

$$\boxed{\dot{a} = \frac{2\sqrt{a}\;\Gamma}{m\sqrt{GM}}}$$

---

### Below the blue lines — Sub-Keplerian gas velocity and integration note

$a(t) = \int dt'\;\dot{a}$ (i.e. the integrator handles this; we only need to supply the instantaneous acceleration)

The gas orbits at sub-Keplerian speed due to the radial pressure gradient. From radial force balance:

$$\frac{v_\phi^2}{R} = \frac{GM}{R^2} + \frac{1}{\rho}\frac{dP}{dR}$$

Since $dP/dR < 0$, the gas velocity is:

$$v_{\phi,\text{gas}} = v_K\left(1 - n\frac{c_s^2}{v_K^2}\right)^{1/2}, \qquad n \equiv -\frac{d\ln P}{d\ln R}$$

and $\Omega(R) = v_{\phi,\text{gas}} / R$. This is already implemented in `DiskModel::disk_v` in `disk-model.hpp`.

---
## Part (i): Generic torque $\Gamma$ — circular orbit derivation

The meeting notes show the intermediate step $\Gamma \to \dot{a}$, but SpaceHub's `add_acc_to` interface requires an **acceleration vector**, not a scalar $\dot{a}$. The key insight is that we don't actually need $\dot{a}$ at all — the N-body integrator evolves the orbit automatically once we supply the correct acceleration. We just need to convert the scalar torque into a tangential force.

### Step 1 — Torque to tangential force

A migration torque $\Gamma$ acts purely in the azimuthal (tangential) direction. By definition of torque:

$$\Gamma = R \cdot F_\phi \quad\Longrightarrow\quad F_\phi = \frac{\Gamma}{R}$$

where $R$ is the cylindrical orbital radius and $F_\phi$ is the tangential force on the body.

### Step 2 — Tangential force to acceleration

The tangential acceleration on the migrating body of mass $m$ is:

$$a_\phi = \frac{F_\phi}{m} = \frac{\Gamma}{m\,R}$$

### Step 3 — Construct the azimuthal unit vector

For position $\vec{r} = (x, y, z)$ relative to the central mass, and velocity $\vec{v}$:

$$\hat{L} = \frac{\vec{r} \times \vec{v}}{|\vec{r} \times \vec{v}|} \qquad \text{(orbital angular momentum direction)}$$

$$\hat{R} = \frac{\vec{r}}{|\vec{r}|} \qquad \text{(radial direction)}$$

$$\hat{\phi} = \hat{L} \times \hat{R} \qquad \text{(prograde tangential direction)}$$

For orbits confined to the $x$–$y$ plane (as in SpaceHub's disk setup), this simplifies to:

$$\hat{\phi} = \left(-\frac{y}{R},\; \frac{x}{R},\; 0\right)$$

### Step 4 — Acceleration vector

$$\boxed{\vec{a}_{\text{mig}} = \frac{\Gamma}{m\,R}\;\hat{\phi}}$$

### Sign convention

| Torque sign | Direction | Effect |
|---|---|---|
| $\Gamma < 0$ | Retrograde $\hat{\phi}$ | Body loses angular momentum $\to$ spirals inward |
| $\Gamma > 0$ | Prograde $\hat{\phi}$ | Body gains angular momentum $\to$ spirals outward |

### Consistency check via $\dot{a}$

For a circular orbit ($r = a$, $v_\phi = \sqrt{GM/a}$):

$$F_\phi = \frac{\Gamma}{a}, \qquad \dot{E} = F_\phi v_\phi = \frac{\Gamma}{a}\sqrt{\frac{GM}{a}}$$

From Keplerian energy $E = -GMm/(2a)$:

$$\dot{E} = \frac{GMm}{2a^2}\dot{a} \quad\Longrightarrow\quad \dot{a} = \frac{2a^2}{GMm}\cdot\frac{\Gamma}{a}\sqrt{\frac{GM}{a}} = \frac{2\sqrt{a}\;\Gamma}{m\sqrt{GM}} \quad\checkmark$$

This recovers the boxed formula from the meeting notes.

---
## Part (ii): Type I migration torques

The Type I torque for a body of mass $m$ orbiting a central mass $M$ at radius $R$ in a disk is (see `typeImigrationV2.ipynb`):

$$\Gamma_I = C_I \cdot \frac{H}{R} \cdot \Gamma_0$$

where the normalization torque is:

$$\Gamma_0 = q^2 \Sigma R^4 \Omega^2 \left(\frac{H}{R}\right)^{-3}, \qquad q \equiv \frac{m}{M}$$

and the dimensionless coefficient $C_I$ depends on the disk gradients and thermodynamics:

| Regime | $C_I$ |
|---|---|
| **Isothermal** | $-1.36 - 0.54\nabla_\Sigma - 0.5\nabla_T$ |
| **Adiabatic/Total** | $-2.34 + (0.46 - 0.96\nabla_\Sigma + 1.8\nabla_T)/\gamma$ |

where $\nabla_\Sigma = d\ln\Sigma/d\ln R$ and $\nabla_T = d\ln T/d\ln R$.

### Applying the generic derivation

Substituting $\Gamma_I$ into the result from Part (i):

$$\vec{a}_{\text{mig},I} = \frac{\Gamma_I}{m\,R}\;\hat{\phi} = \frac{C_I}{m\,R} \cdot \frac{H}{R} \cdot q^2 \Sigma R^4 \Omega^2 \left(\frac{H}{R}\right)^{-3} \;\hat{\phi}$$

Simplifying ($R^4/R = R^3$, $(H/R)(H/R)^{-3} = (H/R)^{-2}$):

$$\boxed{\vec{a}_{\text{mig},I} = C_I \, q^2 \, \frac{\Sigma\, R^3\, \Omega^2}{m} \left(\frac{H}{R}\right)^{-2} \;\hat{\phi}}$$

All quantities ($\Sigma$, $H$, $\Omega$, gradients for $C_I$) are interpolated from the disk table at the particle's current cylindrical radius $R$.

### SpaceHub pseudocode

```cpp
// Inside add_acc_to, for particle i orbiting central mass 0:
auto dr = p[i] - p[0];
double R_cyl = sqrt(dr.x*dr.x + dr.y*dr.y);

// Interpolate disk properties
double Sigma  = interp(R_cyl, &DiskRow::Sigma);
double H      = interp(R_cyl, &DiskRow::H);
double Omega  = sqrt(consts::G * m[0] / (R_cyl*R_cyl*R_cyl));
double grad_T = interp(R_cyl, &DiskRow::grad_T);
double grad_S = interp(R_cyl, &DiskRow::grad_Sigma);

// Compute C_I (isothermal example)
double C_I = -1.36 - 0.54*grad_S - 0.5*grad_T;

// Compute Gamma_I
double q_ratio = m[i] / m[0];
double h = H / R_cyl;
double Gamma0 = q_ratio*q_ratio * Sigma * R_cyl*R_cyl*R_cyl*R_cyl
              * Omega*Omega / (h*h*h);
double Gamma_I = C_I * h * Gamma0;

// Apply as tangential acceleration
double a_phi = Gamma_I / (m[i] * R_cyl);
acceleration[i].x += a_phi * (-dr.y / R_cyl);
acceleration[i].y += a_phi * ( dr.x / R_cyl);
// Newton's 3rd law back-reaction on central mass:
acceleration[0].x -= (Gamma_I / (m[0] * R_cyl)) * (-dr.y / R_cyl);
acceleration[0].y -= (Gamma_I / (m[0] * R_cyl)) * ( dr.x / R_cyl);
```

Note: This is fundamentally different from the drag forces in `disk-model.hpp`, which act anti-parallel to $\vec{v}_\text{rel}$. Migration torques act **tangential to the orbit** in the $\hat{\phi}$ direction.

---
## Part (iii): Eccentric orbits — Fairbairn & Rafikov (2025)

*Reference: Fairbairn & Rafikov, MNRAS 537, 1779 (2025); arXiv:2407.20398*

For eccentric orbits, the circular-orbit derivation above breaks down because the angular momentum budget depends on both $a$ and $e$. F&R 2025 provide numerically computed torques across a grid of disk parameters $(q, p, h_p)$ and eccentricities, expressed as inverse orbital evolution timescales.

### The F&R framework

**Angular momentum** of an eccentric Keplerian orbit:

$$L = M_p \sqrt{GM_* a(1-e^2)} = M_p \Omega_p a^2 \sqrt{1-e^2}$$

Taking the time derivative:

$$\frac{1}{L}\frac{dL}{dt} = \frac{1}{2a}\frac{da}{dt} - \frac{e}{1-e^2}\frac{de}{dt}$$

**Inverse timescale definitions** (following Ida et al. 2020):

$$\tau_L^{-1} = -\frac{1}{L}\frac{dL}{dt}, \qquad \tau_a^{-1} = -\frac{1}{a}\frac{da}{dt}, \qquad \tau_e^{-1} = -\frac{1}{e}\frac{de}{dt}$$

These are linked by:

$$\tau_L^{-1} = \frac{1}{2}\tau_a^{-1} - \frac{e^2}{1-e^2}\tau_e^{-1}$$

**Mapping torques to timescales:**

The net disk torque acts back on the planet as $dL/dt = -T_\text{net}$, so:

$$\tau_L^{-1} = \frac{T_\text{net}}{L}$$

For the semi-major axis, each $(m,l)$ Fourier mode of the eccentric potential rotates at pattern speed $\omega_{ml} = (l/m)\Omega_p$ and conserves the Jacobi integral $E_J = E - \omega_{ml}L$. This gives $dE/dt = -\omega_{ml}T_{\text{net},ml}$, and since $E = -GM_* M_p/(2a)$:

$$\tau_a^{-1} = \frac{2}{M_p a^2 \Omega_p^2} \sum_{m,l} \omega_{ml}\, T_{\text{net},ml}$$

Finally, $\tau_e^{-1}$ is obtained by rearranging the timescale relation above.

**Characteristic scales** used for normalization:

$$F_{J,0} = \Sigma_p\, a^4\, \Omega_p^2\, h_p^{-3} \left(\frac{M_p}{M_*}\right)^2, \qquad \tau_0^{-1} = \frac{F_{J,0}}{M_p \Omega_p a^2}$$

Note that $F_{J,0}$ is the same normalization as $\Gamma_0$ from the Type I torques. The paper provides $\tau_L^{-1}/\tau_0^{-1}$, $\tau_a^{-1}/\tau_0^{-1}$, and $\tau_e^{-1}/(100\,\tau_0^{-1})$ as functions of $e$ for each $(q, p, h_p)$ combination.

### Key physical result

The torque **reverses sign** near $\tilde{e} \equiv e/h_p \sim 1$ (the transonic crossing), but this angular momentum reversal does **not** immediately reverse migration direction. Rapid eccentricity damping ($\sim 10^2 \times$ faster than migration) can allow $L$ to increase even while $a$ decreases.

### Deriving acceleration vectors from $\tau_a^{-1}$ and $\tau_e^{-1}$

Unlike Part (i) where we had a single tangential torque, the eccentric case requires **two independent acceleration components** to separately control $\dot{a}$ and $\dot{e}$.

#### The problem

We are given orbit-averaged rates:

$$\frac{\dot{a}}{a} = -\frac{1}{\tau_a}, \qquad \frac{\dot{e}}{e} = -\frac{1}{\tau_e}$$

and need instantaneous acceleration vectors for `add_acc_to`. The standard approach (Papaloizou & Larwood 2000; Cresswell & Nelson 2008) decomposes the acceleration into a **migration** component (changes energy / semi-major axis) and an **eccentricity damping** component (circularizes the orbit).

#### Step 1 — Migration acceleration (semi-major axis evolution)

For a Keplerian orbit, the energy is $E = -GMm/(2a)$, so:

$$\frac{\dot{E}}{E} = -\frac{\dot{a}}{a} = \frac{1}{\tau_a}$$

A non-Keplerian acceleration $\vec{a}_\text{mig}$ does work at rate $\dot{E}/m = \vec{a}_\text{mig} \cdot \vec{v}$ (see the work-energy derivation in the previous session). Therefore:

$$\frac{\dot{a}}{a} = -\frac{\dot{E}}{E} = -\frac{m\,\vec{a}_\text{mig}\cdot\vec{v}}{E} = \frac{2a}{GM}\,\vec{a}_\text{mig}\cdot\vec{v}$$

Setting $\dot{a}/a = -1/\tau_a$:

$$\vec{a}_\text{mig}\cdot\vec{v} = -\frac{GM}{2a\tau_a}$$

For a circular orbit $v^2 = GM/a$, so the right-hand side is $-v^2/(2\tau_a)$. Now **assume** $\vec{a}_\text{mig}$ is parallel (or anti-parallel) to $\vec{v}$ — i.e. write $\vec{a}_\text{mig} = \alpha\,\vec{v}$. Then:

$$\alpha\, v^2 = -\frac{v^2}{2\tau_a} \quad\Longrightarrow\quad \alpha = -\frac{1}{2\tau_a}$$

$$\boxed{\vec{a}_{\text{mig}} = -\frac{\vec{v}}{2\tau_a}}$$

This is the Papaloizou & Larwood (2000) prescription. It is an **ansatz** (modeling choice), not a unique derivation: any $\vec{a}$ with the correct projection onto $\vec{v}$ would reproduce the same $\dot{a}/a$. The parallel-to-$\vec{v}$ choice is adopted because $\vec{v}$ naturally rotates with the orbit and encodes the correct phase-dependent energy injection rate $\dot{E}/m = -v^2/(2\tau_a)$ at every orbital phase. For eccentric orbits, $v^2 = GM(2/r - 1/a)$ varies correctly around the orbit via the vis-viva relation, so the orbit-averaged $\langle\dot{a}/a\rangle$ is automatically correct to leading order. A purely tangential ($\hat{\phi}$) force with constant amplitude would *not* yield the correct orbit-averaged $\dot{a}$ for $e \neq 0$ without introducing an explicit phase-dependent modulation.

**Verification (circular limit):** $\dot{E}/m = -v^2/(2\tau_a)$. With $v^2 = GM/a$ and $E/m = -GM/(2a)$: $\dot{a}/a = -\dot{E}/E = -(- v^2/(2\tau_a)) / (-GM/(2a)) = -1/\tau_a$. $\checkmark$

Note that this is a **velocity-dependent** force (drag-like). For inward migration ($\tau_a > 0$), it opposes the velocity and removes orbital energy.

#### Step 2 — Eccentricity damping acceleration

Eccentricity manifests as radial velocity oscillations around the circular value. To damp $e$ without (to leading order) affecting $a$, we apply an acceleration that damps the **radial component** of velocity:

$$\boxed{\vec{a}_{\text{ecc}} = -\frac{2\,(\vec{v} \cdot \hat{r})}{\tau_e}\,\hat{r}}$$

where $\hat{r} = \vec{r}/|\vec{r}|$ is the radial unit vector (relative to the central mass).

**Why this works:** For a Keplerian orbit, $v_r = e\sin f \cdot na/\sqrt{1-e^2}$ where $f$ is the true anomaly and $n = \sqrt{GM/a^3}$. The radial acceleration $a_r = -2v_r/\tau_e$ enters the Gauss planetary equation for $\dot{e}$:

$$\dot{e} = \frac{\sqrt{1-e^2}}{na}\sin f \cdot a_r + \ldots$$

Substituting $a_r = -2v_r/\tau_e = -2nae\sin f/(\sqrt{1-e^2}\,\tau_e)$:

$$\dot{e}\big|_\text{radial} = -\frac{2e\sin^2 f}{\tau_e}$$

Orbit-averaging with $\langle \sin^2 f \rangle = 1/2$ gives $\langle \dot{e} \rangle / e = -1/\tau_e$. $\checkmark$

**Cross-coupling:** This acceleration also affects $\dot{a}$ via the Gauss equation, contributing $\langle\dot{a}/a\rangle = -2e^2/((1-e^2)\tau_e)$, which is $O(e^2)$ and subdominant for $e \lesssim h_p$.

#### Step 3 — Total acceleration

$$\boxed{\vec{a}_{\text{total}} = -\frac{\vec{v}}{2\tau_a} - \frac{2\,(\vec{v} \cdot \hat{r})}{\tau_e}\,\hat{r}}$$

This is the complete prescription for `add_acc_to` when using orbit-averaged timescales.

#### Important caveats

- The two components are not perfectly decoupled: the migration term slightly affects $e$, and vice versa (see cross-coupling note above). But for $e \lesssim h_p$ these cross-terms are subdominant.
- These are **orbit-averaged** prescriptions applied **instantaneously**. This is valid when the evolution timescale $\gg$ the orbital period, which is the regime where the F&R torques are computed.
- For the circular limit ($e \to 0$), $\vec{v}\cdot\hat{r} \to 0$, so the eccentricity term vanishes and we recover $\vec{a}_\text{mig} = -\vec{v}/(2\tau_a)$, consistent with the tangential torque from Part (i).

### Evaluating $\tau_a^{-1}$ and $\tau_e^{-1}$ from F&R data

F&R 2025 provide their numerical results as a data cube over $(e, q, p, h_p)$. The procedure to get dimensional timescales is:

**1. Compute the characteristic timescale** at the particle's current orbital radius $R$:

$$\tau_0^{-1}(R) = \frac{\Sigma(R)\, R^4\, \Omega(R)^2\, h(R)^{-3}}{M_p \Omega(R) R^2} \cdot \left(\frac{M_p}{M_*}\right)^2 = \frac{M_p}{M_*^2} \cdot \Sigma(R)\, R^2\, \Omega(R) \cdot h(R)^{-3}$$

(Here $\Sigma$, $h = H/R$, and $\Omega$ come from your disk model, and $M_p$, $M_*$ are the particle and central masses.)

**2. Interpolate the normalized rates** from the F&R data cube at the current $(e, q, p, h)$:

$$\left(\frac{\tau_a^{-1}}{\tau_0^{-1}}\right)_{\text{F\&R}}, \qquad \left(\frac{\tau_e^{-1}}{100\,\tau_0^{-1}}\right)_{\text{F\&R}}$$

**3. Recover dimensional inverse timescales:**

$$\tau_a^{-1} = \left(\frac{\tau_a^{-1}}{\tau_0^{-1}}\right)_{\text{F\&R}} \cdot \tau_0^{-1}(R)$$

$$\tau_e^{-1} = 100 \cdot \left(\frac{\tau_e^{-1}}{100\,\tau_0^{-1}}\right)_{\text{F\&R}} \cdot \tau_0^{-1}(R)$$

**4. Plug into the acceleration prescription** from the previous section.

### Alternative: Using fitting functions

F&R note that simple fitting functions across the full $(q, p, h_p)$ parameter space are difficult. However, earlier fitting formulas (valid for specific disk profiles) can be used as approximations. Two commonly cited prescriptions:

**Papaloizou & Larwood (2000)** — for $q = 1.0$, $p = 1.5$:

$$\tau_{L,\text{PL00}}^{-1} = 7.33\left(\frac{b}{0.5}\right)^{-1.75} h_p \frac{1 - (\tilde{e}/1.1)^4}{1 + (\tilde{e}/1.3)^5}\;\tau_0^{-1}$$

$$\tau_{e,\text{PL00}}^{-1} = 4.26\left(\frac{b}{0.5}\right)^{-2.5} \left(1 + \frac{1}{4}\tilde{e}^3\right)^{-1} h_p^{-1}\;\tau_0^{-1}$$

**Cresswell & Nelson (2006)** — calibrated to 2D simulations:

$$\tau_{L,\text{CN06}}^{-1} = \frac{2.7 + 1.1p}{2}\, h_p \frac{1 - (\tilde{e}/1.1)^4}{1 + (\tilde{e}/1.3)^5}\;\tau_0^{-1}$$

$$\tau_{e,\text{CN06}}^{-1} = \frac{0.78}{Q_e}\left(1 + \frac{1}{4}\tilde{e}^3\right)^{-1} h_p^{-1}\;\tau_0^{-1}$$

where $\tilde{e} = e/h_p$, $b$ is the gravitational softening parameter, and $Q_e \sim 0.1$–$1.0$.

To convert these $\tau_L^{-1}$ prescriptions to $\tau_a^{-1}$ (which is what the acceleration formula needs), use the timescale relation:

$$\tau_a^{-1} = 2\tau_L^{-1} + \frac{2e^2}{1-e^2}\tau_e^{-1}$$

### SpaceHub pseudocode for eccentric migration

```cpp
// Inside add_acc_to for DiskMigration, for particle i orbiting central mass 0:
auto dr = p[i] - p[0];
auto dv = v[i] - v[0];
double R_cyl = sqrt(dr.x*dr.x + dr.y*dr.y);
double r_mag = sqrt(dot(dr, dr));

// Compute current orbital elements from state vectors
double v2 = dot(dv, dv);
double mu = consts::G * (m[0] + m[i]);
double a_orb = -mu / (v2 - 2*mu/r_mag);  // semi-major axis from vis-viva
// ... compute e from angular momentum and energy ...

// Interpolate disk properties at R_cyl
double Sigma = interp(R_cyl, &DiskRow::Sigma);
double H     = interp(R_cyl, &DiskRow::H);
double h     = H / R_cyl;
double Omega = sqrt(consts::G * m[0] / (R_cyl*R_cyl*R_cyl));

// Characteristic inverse timescale
double q_ratio = m[i] / m[0];
double tau0_inv = q_ratio*q_ratio * Sigma * R_cyl*R_cyl * Omega / (m[i] * h*h*h);

// Get normalized inverse timescales from F&R data cube
// (interpolate in e, q, p, h)
double tau_a_inv = lookup_tau_a_inv(e, q_disk, p_disk, h) * tau0_inv;
double tau_e_inv = lookup_tau_e_inv(e, q_disk, p_disk, h) * tau0_inv;
// OR: use PL00/CN06 fitting functions directly

// Compute acceleration components
// Migration: a_mig = -v / (2 * tau_a)
auto a_mig_x = -dv.x / (2.0 / tau_a_inv);
auto a_mig_y = -dv.y / (2.0 / tau_a_inv);
auto a_mig_z = -dv.z / (2.0 / tau_a_inv);
// Simplifies to:
// a_mig = -dv * tau_a_inv / 2

// Eccentricity damping: a_ecc = -2 * (v . r_hat) * r_hat / tau_e
double vr = dot(dv, dr) / r_mag;  // radial velocity
auto r_hat_x = dr.x / r_mag;
auto r_hat_y = dr.y / r_mag;
auto r_hat_z = dr.z / r_mag;
// a_ecc = -2 * vr * r_hat * tau_e_inv

// Apply total acceleration
acceleration[i].x += -dv.x * tau_a_inv * 0.5 - 2.0 * vr * r_hat_x * tau_e_inv;
acceleration[i].y += -dv.y * tau_a_inv * 0.5 - 2.0 * vr * r_hat_y * tau_e_inv;
acceleration[i].z += -dv.z * tau_a_inv * 0.5 - 2.0 * vr * r_hat_z * tau_e_inv;
// Back-reaction on central mass (Newton's 3rd law)
acceleration[0] -= (m[i]/m[0]) * (acceleration change on i);
```

### Summary of the three regimes

| Regime | Input | Acceleration formula | Force character |
|---|---|---|---|
| **(i) Generic $\Gamma$, circular** | Scalar torque $\Gamma(R)$ | $\vec{a} = \frac{\Gamma}{mR}\hat{\phi}$ | Tangential only |
| **(ii) Type I, circular** | $C_I$, $\Gamma_0$ from disk model | Same as (i) with $\Gamma = C_I (H/R) \Gamma_0$ | Tangential only |
| **(iii) Eccentric (F&R)** | $\tau_a^{-1}$, $\tau_e^{-1}$ | $\vec{a} = -\frac{\vec{v}}{2\tau_a} - \frac{2(\vec{v}\cdot\hat{r})}{\tau_e}\hat{r}$ | Velocity-dependent (both tangential and radial) |

For **circular orbits**, regimes (i) and (iii) are equivalent: setting $e = 0$ in (iii) kills the eccentricity damping term, and the migration term $-\vec{v}/(2\tau_a)$ reduces to $\Gamma/(mR)\,\hat{\phi}$ via the relation $\tau_a^{-1} = -2\Gamma/(mR v_\phi)$.